In [1]:

# PROJECT 4: CONTACT BOOK
# DecodeLabs Industrial Training Kit | Batch 2026


# IMPORTS


import re  # Regular expressions — phone number aur email validate karne ke liye


# =============================================================================
# STEP 1: DATA STRUCTURE — Nested Dictionary (The Core)
# =============================================================================
# contacts = {
#     "Ali Hassan": {
#         "phone": "03001234567",
#         "email": "ali@gmail.com",
#         "id"   : 1
#     },
#     "Sara Khan": {
#         "phone": "03007654321",
#         "email": "sara@gmail.com",
#         "id"   : 2
#     }
# }
#
# Outer dictionary = Full database table
# Key (name)       = Primary Key (unique identifier)
# Value (dict)     = Row data (phone, email)
#
# Dictionary lookup = O(1) — hash table ki wajah se instant access!
# List search       = O(N) — har element check karna padta hai
# Dictionary >> List for data retrieval
# =============================================================================

contacts      = {}   # Khali nested dictionary — hamara contact book
contact_id    = 0    # Auto-increment ID counter — har naye contact ke liye

# STEP 2: VALIDATION FUNCTIONS — Defensive Coding (Phase 1: Input)

def validate_phone(phone):
    # Phone number validate karo — sirf digits, 10-13 characters
    # re.match() = Regular Expression — pattern matching tool
    # r'^\d{10,13}$' matlab:
    #   ^ = string ki shuru se
    #   \d = sirf digits (0-9)
    #   {10,13} = 10 se 13 characters
    #   $ = string ka end
    pattern = r'^\d{10,13}$'
    return bool(re.match(pattern, phone))


def validate_email(email):
    # Basic email validate karo — @ aur . hona chahiye
    # r'^[\w.-]+@[\w.-]+\.\w+$' matlab:
    #   [\w.-]+ = letters, digits, dots, hyphens (ek ya zyada)
    #   @       = @ sign zaroor hona chahiye
    #   \.      = dot (escaped)
    #   \w+     = domain extension (com, pk, org etc)
    pattern = r'^[\w.-]+@[\w.-]+\.\w+$'
    return bool(re.match(pattern, email))

# STEP 3: CREATE — Contact Add Karna (INSERT operation)


def add_contact():
    global contact_id  # Global counter update karenge

    print("\n  ➕ NAYA CONTACT ADD KARO")
    print("  " + "-" * 40)

    # ----- Name Input -----
    name = input("  👤 Naam: ").strip()

    # Name khali nahi hona chahiye
    if not name:
        print("  ❌ Naam khali nahi ho sakta!\n")
        return

    # Case-insensitive check — "ali" aur "Ali" same hain
    # .lower() se sab lowercase kar ke compare karte hain
    for existing_name in contacts.keys():
        if existing_name.lower() == name.lower():
            print(f"  ❌ '{name}' already exist karta hai!\n")
            return

    # ----- Phone Input -----
    phone = input("  📞 Phone (10-13 digits): ").strip()

    # Phone validation — sirf numbers, sahi length
    if not validate_phone(phone):
        print("  ❌ Invalid phone! Sirf digits, 10-13 characters.\n")
        return

    # ----- Email Input -----
    email = input("  📧 Email: ").strip()

    # Email validation — @ aur . hona chahiye
    if not validate_email(email):
        print("  ❌ Invalid email format! (e.g. name@gmail.com)\n")
        return

    # ----- Contact Dictionary Banao -----
    # Yeh ek "row" hai hamare in-memory database mein
    contact_id += 1  # ID increment karo — Primary Key

    # Nested dictionary mein store karo
    # Key = name (unique identifier)
    # Value = contact details dictionary
    contacts[name] = {
        "phone": phone,   # Phone number
        "email": email,   # Email address
        "id"   : contact_id  # Unique ID
    }

    print(f"\n  ✅ Contact add ho gaya: '{name}' (ID: {contact_id})\n")


# STEP 4: READ — Saare Contacts Dikhana (SELECT * operation)


def view_all_contacts():
    # Check karo ke contacts hain ya nahi
    if not contacts:  # Empty dictionary = False
        print("\n  📭 Koi contact nahi hai! Pehle add karo.\n")
        return

    print("\n" + "=" * 60)
    print("  📒 AAPKA CONTACT BOOK")
    print("=" * 60)

    # .items() = key aur value dono ek saath milte hain
    # Yeh enumerate() jaisa hai — professional Pythonic way
    for index, (name, details) in enumerate(contacts.items(), start=1):
        print(f"\n  {index}. 👤 {name}")
        print(f"     📞 Phone : {details['phone']}")
        print(f"     📧 Email : {details['email']}")
        print(f"     🆔 ID    : {details['id']}")
        print("     " + "-" * 35)

    print(f"\n  📊 Total Contacts: {len(contacts)}")  # len() = count
    print("=" * 60 + "\n")

# STEP 5: SEARCH — Contact Dhundna (SELECT WHERE operation)


def search_contact():
    if not contacts:
        print("\n  📭 Koi contact nahi hai!\n")
        return

    # Search query lo
    query = input("\n  🔍 Naam likho (ya naam ka hissa): ").strip().lower()

    if not query:
        print("  ❌ Kuch toh likho search karne ke liye!\n")
        return

    # Search results collect karo
    results = []

    # .items() se har contact check karo
    for name, details in contacts.items():
        # .lower() se case-insensitive search
        # 'ali' in 'ali hassan' = True  ← partial search bhi kaam karta hai!
        if query in name.lower():
            results.append((name, details))  # Match mila — results mein add karo

    # Results display karo
    if results:
        print(f"\n  🔍 '{query}' ke liye {len(results)} result(s) mile:\n")
        print("=" * 60)

        for name, details in results:
            print(f"\n  👤 {name}")
            print(f"     📞 Phone : {details['phone']}")
            print(f"     📧 Email : {details['email']}")
            print("     " + "-" * 35)

        print("=" * 60 + "\n")
    else:
        # Koi match nahi mila
        print(f"\n  ❌ '{query}' naam ka koi contact nahi mila.\n")


# STEP 6: UPDATE — Contact Update Karna (UPDATE SET operation)


def update_contact():
    if not contacts:
        print("\n  📭 Koi contact nahi hai!\n")
        return

    view_all_contacts()  # Pehle saare contacts dikhao

    # Konsa contact update karna hai?
    name = input("  ✏️  Konsa contact update karein? (naam likho): ").strip()

    # Case-insensitive search — exact naam dhundo
    found_name = None
    for existing_name in contacts.keys():
        if existing_name.lower() == name.lower():
            found_name = existing_name  # Original naam (correct case ke saath)
            break

    if not found_name:
        print(f"\n  ❌ '{name}' naam ka contact nahi mila!\n")
        return

    print(f"\n  ✏️  '{found_name}' update ho raha hai...")
    print("  (Enter dabao purana value rakhne ke liye)\n")

    # ----- Phone Update -----
    current_phone = contacts[found_name]['phone']
    new_phone = input(f"  📞 Naya Phone [{current_phone}]: ").strip()

    # Agar khali hai toh purana rakho
    if new_phone:
        if not validate_phone(new_phone):
            print("  ❌ Invalid phone format!\n")
            return
    else:
        new_phone = current_phone  # Purana value preserve karo

    # ----- Email Update -----
    current_email = contacts[found_name]['email']
    new_email = input(f"  📧 Naya Email [{current_email}]: ").strip()

    if new_email:
        if not validate_email(new_email):
            print("  ❌ Invalid email format!\n")
            return
    else:
        new_email = current_email  # Purana value preserve karo

    # ----- Dictionary Update karo -----
    # Dictionary mutable hai — directly update ho jaata hai
    # Sirf changed fields update karo — ID preserve karo
    contacts[found_name]['phone'] = new_phone
    contacts[found_name]['email'] = new_email

    print(f"\n  ✅ '{found_name}' successfully update ho gaya!\n")


# STEP 7: DELETE — Contact Delete Karna (DELETE FROM operation)


def delete_contact():
    if not contacts:
        print("\n  📭 Koi contact nahi hai!\n")
        return

    view_all_contacts()  # Pehle dikhao

    # Konsa delete karna hai?
    name = input("  🗑️  Konsa contact delete karein? (naam likho): ").strip()

    # Case-insensitive search
    found_name = None
    for existing_name in contacts.keys():
        if existing_name.lower() == name.lower():
            found_name = existing_name
            break

    if not found_name:
        print(f"\n  ❌ '{name}' naam ka contact nahi mila!\n")
        return

    # Confirm karo — accidental deletion se bachao
    confirm = input(f"\n  ⚠️  '{found_name}' delete karna chahte ho? (yes/no): ").strip().lower()

    if confirm == 'yes':
        # del keyword — dictionary se key-value pair remove karta hai
        del contacts[found_name]
        print(f"\n  🗑️  '{found_name}' delete ho gaya!\n")
    else:
        # User ne cancel kiya
        print("  ↩️  Delete cancel ho gaya.\n")


# STEP 8: STATISTICS — Bonus Feature


def show_stats():
    if not contacts:
        print("\n  📭 Koi contact nahi hai!\n")
        return

    # Gmail users count karo — dictionary values iterate karo
    gmail_count = sum(
        1 for details in contacts.values()
        if 'gmail.com' in details['email']
    )

    print("\n" + "=" * 40)
    print("  📊 CONTACT BOOK STATISTICS")
    print("=" * 40)
    print(f"  Total Contacts : {len(contacts)}")
    print(f"  Gmail Users    : {gmail_count}")
    print(f"  Other Emails   : {len(contacts) - gmail_count}")

    # .keys() se saare naam dikhao
    print(f"\n  All Names:")
    for name in contacts.keys():  # .keys() = sirf keys (names)
        print(f"    • {name}")

    print("=" * 40 + "\n")

# STEP 9: MAIN FUNCTION — Complete IPO Engine


def main():
    print("\n" + "=" * 60)
    print("  📒 DECODELABS CONTACT BOOK")
    print("  Batch 2026 | Python Project 4")
    print("  Dictionary-Powered | CRUD Operations")
    print("=" * 60)

    while True:
        print("\n  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
        print("  Kya karna chahte ho?")
        print("  1. ➕ Contact add karo")
        print("  2. 📋 Saare contacts dekho")
        print("  3. 🔍 Contact search karo")
        print("  4. ✏️  Contact update karo")
        print("  5. 🗑️  Contact delete karo")
        print("  6. 📊 Statistics dekho")
        print("  7. 🚪 Exit")
        print("  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

        choice = input("  Option chuno (1-7): ").strip()

        if   choice == "1": add_contact()
        elif choice == "2": view_all_contacts()
        elif choice == "3": search_contact()
        elif choice == "4": update_contact()
        elif choice == "5": delete_contact()
        elif choice == "6": show_stats()
        elif choice == "7":
            print("\n  👋 Contact Book band ho raha hai...")
            print("  ⚠️  Note: RAM volatile — data save nahi hoga!")
            print("  DecodeLabs | Keep Coding! 💪\n")
            break
        else:
            print("\n  ⚠️  1 se 7 ke beech option chuno!\n")


if __name__ == "__main__":
    main()



  📒 DECODELABS CONTACT BOOK
  Batch 2026 | Python Project 4
  Dictionary-Powered | CRUD Operations

  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Kya karna chahte ho?
  1. ➕ Contact add karo
  2. 📋 Saare contacts dekho
  3. 🔍 Contact search karo
  4. ✏️  Contact update karo
  5. 🗑️  Contact delete karo
  6. 📊 Statistics dekho
  7. 🚪 Exit
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


  Option chuno (1-7):  1



  ➕ NAYA CONTACT ADD KARO
  ----------------------------------------


  👤 Naam:  Haroon
  📞 Phone (10-13 digits):  03009048571
  📧 Email:  haroonrasheedlakho@gmail.com



  ✅ Contact add ho gaya: 'Haroon' (ID: 1)


  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Kya karna chahte ho?
  1. ➕ Contact add karo
  2. 📋 Saare contacts dekho
  3. 🔍 Contact search karo
  4. ✏️  Contact update karo
  5. 🗑️  Contact delete karo
  6. 📊 Statistics dekho
  7. 🚪 Exit
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


  Option chuno (1-7):  3

  🔍 Naam likho (ya naam ka hissa):  haroon



  🔍 'haroon' ke liye 1 result(s) mile:


  👤 Haroon
     📞 Phone : 03009048571
     📧 Email : haroonrasheedlakho@gmail.com
     -----------------------------------


  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Kya karna chahte ho?
  1. ➕ Contact add karo
  2. 📋 Saare contacts dekho
  3. 🔍 Contact search karo
  4. ✏️  Contact update karo
  5. 🗑️  Contact delete karo
  6. 📊 Statistics dekho
  7. 🚪 Exit
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


  Option chuno (1-7):  7



  👋 Contact Book band ho raha hai...
  ⚠️  Note: RAM volatile — data save nahi hoga!
  DecodeLabs | Keep Coding! 💪

